# Prueba de Versión Mejorada

Este notebook prueba la versión mejorada de `get_company_count()` con mejor búsqueda y validación.

In [ ]:
import pandas as pd
import requests
import time

# Headers necesarios para que Wikidata acepte las solicitudes
HEADERS = {
    'User-Agent': 'DataMiningProject/1.0 (Educational Purpose)',
    'Accept': 'application/json'
}

def clean_name(name):
    """Limpia el nombre eliminando ' & family' y otros sufijos"""
    name = name.replace(" & family", "")
    name = name.strip()
    return name

def is_business_person(entity_data, entity_id):
    """Verifica si la entidad es un empresario/billonario"""
    claims = entity_data["entities"][entity_id].get("claims", {})
    
    # P106 = occupation (ocupación)
    if "P106" in claims:
        for claim in claims["P106"]:
            try:
                occupation_id = claim["mainsnak"]["datavalue"]["value"]["id"]
                # IDs relevantes: Q131524=entrepreneur, Q43845=businessperson, Q140686=chairperson
                # Q484876=CEO, Q1273818=investor, Q205375=philanthropist
                if occupation_id in ["Q131524", "Q43845", "Q140686", "Q484876", "Q1273818", "Q205375"]:
                    return True
            except (KeyError, TypeError):
                continue
    
    # Verificar descripción
    descriptions = entity_data["entities"][entity_id].get("descriptions", {})
    desc_en = descriptions.get("en", {}).get("value", "").lower()
    business_keywords = ["business", "entrepreneur", "billionaire", "investor", "ceo", "founder"]
    if any(keyword in desc_en for keyword in business_keywords):
        return True
    
    return False

def get_company_count_improved(name, max_retries=3):
    """Obtiene el número de compañías asociadas a una persona desde Wikidata (versión mejorada)"""
    
    # Limpiar el nombre
    clean = clean_name(name)
    
    for attempt in range(max_retries):
        try:
            # Paso 1: Buscar la entidad de la persona (buscar más resultados)
            search_url = "https://www.wikidata.org/w/api.php"
            search_params = {
                "action": "wbsearchentities",
                "search": clean,
                "language": "en",
                "format": "json",
                "type": "item",
                "limit": 5  # Buscar hasta 5 resultados
            }
            
            search_response = requests.get(
                search_url, 
                params=search_params, 
                headers=HEADERS,
                timeout=30
            )
            search_response.raise_for_status()
            search_data = search_response.json()
            
            if not search_data.get("search"):
                print(f"  ⚠️ No se encontró entidad para: {clean}")
                return 0
            
            # Intentar encontrar la persona correcta (empresario/billonario)
            entity_id = None
            label = None
            
            for result in search_data["search"]:
                candidate_id = result["id"]
                candidate_label = result.get('label', '')
                candidate_desc = result.get('description', '').lower()
                
                # Verificar si la descripción sugiere que es un empresario
                business_keywords = ["business", "entrepreneur", "billionaire", "investor", "ceo", "founder", "chairperson"]
                if any(keyword in candidate_desc for keyword in business_keywords):
                    entity_id = candidate_id
                    label = candidate_label
                    print(f"  ✓ Encontrado: {entity_id} - {label} ({candidate_desc})")
                    break
            
            # Si no encontramos uno obvio, usar el primero
            if not entity_id:
                entity_id = search_data["search"][0]["id"]
                label = search_data["search"][0].get('label', '')
                print(f"  ⚠️ Usando primer resultado: {entity_id} - {label}")
            
            # Paso 2: Obtener datos de la entidad
            entity_url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"
            entity_response = requests.get(
                entity_url, 
                headers=HEADERS,
                timeout=30
            )
            entity_response.raise_for_status()
            entity_data = entity_response.json()
            
            claims = entity_data["entities"][entity_id].get("claims", {})
            
            # Verificar si es empresario (ayuda a confirmar identidad)
            if is_business_person(entity_data, entity_id):
                print(f"  ✓ Confirmado como empresario/inversor")
            
            # Propiedades de Wikidata relacionadas con compañías (expandidas)
            company_properties = {
                "P112": "founder of",           # fundador de
                "P169": "chief executive officer", # CEO de
                "P488": "chairperson",          # presidente de
                "P1037": "director/manager",    # director/gerente de
                "P108": "employer",             # empleador
                "P1830": "owner of",            # propietario/inversor de
                "P127": "owned by",             # propiedad de (inverso, para verificar)
                "P1454": "legal form",          # forma legal (para holdings)
                "P749": "parent organization"   # organización matriz
            }
            
            companies = set()
            property_counts = {}
            
            # Recolectar todas las compañías/organizaciones
            for prop_id, prop_name in company_properties.items():
                if prop_id in claims:
                    prop_count = 0
                    for claim in claims[prop_id]:
                        try:
                            # Obtener el ID de la compañía
                            company_id = claim["mainsnak"]["datavalue"]["value"]["id"]
                            companies.add(company_id)
                            prop_count += 1
                        except (KeyError, TypeError):
                            continue
                    if prop_count > 0:
                        property_counts[prop_name] = prop_count
            
            count = len(companies)
            
            # Logging detallado
            if count > 0:
                print(f"  → Compañías encontradas: {count}")
                if property_counts:
                    print(f"     Desglose: {', '.join([f'{k}: {v}' for k, v in property_counts.items()])}")
            else:
                print(f"  ⚠️ 0 compañías - revisar manualmente")
            
            return count
            
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                print(f"  ⏱ Timeout, reintentando ({attempt + 1}/{max_retries})...")
                time.sleep(2)
                continue
            else:
                print(f"  ✗ Timeout después de {max_retries} intentos")
                return 0
                
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                print(f"  ⚠️ Error: {str(e)[:50]}, reintentando...")
                time.sleep(2)
                continue
            else:
                print(f"  ✗ Error de conexión: {str(e)[:100]}")
                return 0
                
        except Exception as e:
            print(f"  ✗ Error: {str(e)[:100]}")
            return 0
    
    return 0

## Prueba con casos problemáticos

Vamos a probar con algunos billonarios que antes daban 0 empresas:

In [ ]:
# Casos que antes dieron 0:
test_cases = [
    "Bernard Arnault",
    "Gautam Adani",
    "Zhong Shanshan",
    "Francoise Bettencourt Meyers"
]

print("Probando casos problemáticos:\n")
print("="*60)

for name in test_cases:
    print(f"\n{name}:")
    count = get_company_count_improved(name)
    print(f"  TOTAL: {count} empresas\n")
    time.sleep(2)

## Comparar con resultados anteriores

In [ ]:
# Cargar resultados anteriores
old_results = pd.read_csv("person_companies.csv")

# Seleccionar una muestra de personas con 0 empresas para re-probar
zero_company_persons = old_results[old_results['numberOfCompanies'] == 0].head(20)

print(f"Re-evaluando {len(zero_company_persons)} personas que tenían 0 empresas:\n")
print("="*60)

new_counts = []
for idx, row in zero_company_persons.iterrows():
    name = row['personName']
    print(f"\n{name}:")
    count = get_company_count_improved(name)
    new_counts.append({'personName': name, 'old_count': 0, 'new_count': count})
    time.sleep(2)

# Crear DataFrame de comparación
comparison = pd.DataFrame(new_counts)
print("\n" + "="*60)
print("\nRESUMEN DE MEJORAS:")
print(comparison)
print(f"\nPersonas que ahora tienen empresas: {(comparison['new_count'] > 0).sum()}/{len(comparison)}")